<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h1>Notebook 2: Markov Chain Monte Carlo — Tutorial Session</h1>

<h2>Structure</h2>

| Part | Topic | Type |
|:----:|-------|------|
| 1 | Quick Recap | Conceptual |
| 2 | NumPyro Crash Course | Hands-on |
| 3 | Two-Planet RV fit | Exercise |
| 4 | Diagnostics | Conceptual + Hands-on |
| 5 | Cepheid P-L | Exercise |
| 6 | Halaxy Brightness | Exercise |


<h2>Preamble</h2>

- You will be required to write code. Skeleton functions with `TODO` comments are provided, along with hint blocks.
- Imports: Run the cell below before starting to ensure you have imported everything correctly

</div>

In [1]:
import numpy as np

# plotting code
import matplotlib.pyplot as plt
import corner

# autodiff capable maths library
import jax
import jax.numpy as jnp
from jax import random

# MCMC Stuff
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS, HMC, Predictive, init_to_value, SA
from numpyro.diagnostics import print_summary

# Helpful functions and plotting code
import arviz as az
from astropy.timeseries import LombScargle

# tell it to use 4 cores, so it can run in parralell
numpyro.set_host_device_count(4)

# run the below line to see how many cores numpyro can see:
# jax.local_device_count()

# Tell numpyro to use the CPU (simpler to get running). You can set it to 'gpu' if you like
# but we won't need it 
numpyro.set_platform('cpu')
rng_key = random.PRNGKey(42)

In [2]:
print('NumPyro Version: ', numpyro.__version__)
print('JAX Version: ', jax.__version__)

NumPyro Version:  0.20.0
JAX Version:  0.9.0.1


<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

# Part 1: Recap
This is a brief recap of previous tutorials, and specifically the pre-reading for this tutorial. I highly recommend that you look at that before coming here.

## Bayes Theorem

In previous tutorials, we defined Bayes Theorem:
$$P(\theta \mid D) = \frac{\mathcal{L}(\theta) \, \pi(\theta)}{\mathcal{Z}}$$
Recall that each of the are:
- $P(\theta|D)$ - The posterior, i.e. what we believe about our parameters *after* seeing the data.
- $\mathcal{L}(\theta)$ - The likelihood, i.e. the probability of seeing the data given a particular set of parameters (the physics and the noise).
- $\pi(\theta)$ - The prior, i.e. what we believe about our parameters *before* seeing the data
- $\mathcal{Z}$ - The evidence, i.e. the total probability of observing your data, averaged over all parameter values weighted by the prior. In practice, treated as a normalisation constant.

The nice thing about Bayes theorem, is that if we only care about parameter estimation (and not model comparison), we can treat the evidence as a normalisation constant and drop it, leaving:

$$P(\theta \mid D) \propto \mathcal{L}(\theta) \, \pi(\theta)$$

In this tutorial, we don't care about the evidence, except in some rare edge cases.

## Monte Carlo Intergation

This is used to approximate integrals by random sampling. This beats out grid search / analytical analysis once you get past a few dimensions. In higher dimensions, most random samples land outside important regions, leading to inefficient sampling.

## Markov Chains
A sequence where the next state depends only on the current state (memoryless). Under mild conditions (irreducibility and aperiodicity), the chain converges to a unique stationary distribution $\pi(\theta)$, regardless of initialisation. Useful for studying probabilistic systems.

## MCMC
The main idea is to construct a Markov chain whose stationary distribution *is* the posterior. Once the chain has converged (burnt-in), its samples are (correlated) draws from the posterior. Because it's acceptance ratio is $$ \alpha = \text{min}(1, \frac{p(\theta')}{p(\theta)}) $$ which looks at the ratios of posteriors, we don't need to calculate the evidence $\mathcal{Z}$ as it cancels out. 

## Metropolis-(Hastings) Algorithm
This was the first MCMC algorithm. The main idea is that when a step (with step size usually drawn from a Gaussian) $\theta \rightarrow \theta'$ is proposed, you calculate an acceptance ratio: $$ \alpha = \text{min}\left(1, \frac{p(\theta')}{p(\theta)}\right) $$
and you accept that proposal with probability $\alpha$. If the next position has a higher probability density (i.e. $p(\theta') > p(\theta)$), the fraction would be greater than one, so $\alpha=1$ and you move to that position. Likewise, if the proposed point has the same probability, you always move (there is no reason to prefer one region over another with equal probability). If the next position has a lower probability (i.e. $p(\theta') < p(\theta)$), you sometimes move there with the probabilty of moving there being $p(\theta')/p(\theta)$. This rule set allows the chain/walker to fully explore the posterior (given infinite time) and the Markov-chain stationary distribution is the posterior distribution.

This algorithm is simple, but the random guess-and-check strategy scales poorly with higher dimensionality. See the before tutorial for why.

## The Typical Set

When thinking about sampling probability distribution functions, you care about *probability mass*. This is probability density multiplied by the volume. The probability mass is not dominated by the mode (or equivalently the MAP / MLE, e.g. the peak of a Gaussian) which has high density but small volume, nor is it dominated by the tails (large volume, small density). Instead the probability mass is dominated by a thin shell where the density $\times$ volume is maximised. This shell is called the typical set. As the number of dimensions grow, this shell becomes more and more narrow, making it hard to find.

<div style="text-align: center;">
  <img src="Images/Betancourt2018.PNG" style="max-width: 70%; border-radius: 4px;">
    <p style="font-size: 0.75em; color: #888; font-style: italic; margin-top: 0.5em; text-align: center;">
   In high dimensions a probability density, π(q), will concentrate around its mode, but the volume over which we integrate that density, dq, is much larger away from the mode. Contributions to any expectation are determined by the product of density and volume, π(q) dq, which then concentrates in a nearly-singular neighborhood called the typical set (grey).
  </p>
</div>

### Hamiltonian Monte Carlo (HMC)

Surprisingly, the way to deal with the problems induced by high dimensionality is to double the number of dimensions!

HMC adds an auxiliary *momentum* variable for each parameter of parameter space. It exploits the gradient of the parameter space ($\nabla \log p(\theta)$) to simulate Hamiltonian dynamics (yes, like in mechanics). This allows proposals that glide along the typical set rather than stumbling blindly around it, allowing samples that are distant, lowly correlated, and on the typical set with high acceptance rates. 

You can think of the gradient as being a gravitational field of a planet, the typical set being a stable orbit around the planet. In this sense, the momentum parameter is what allows the walker to be in a stable orbit, rather than falling into the planet (following the gradient vector field).

<div style="text-align: center;">
  <img src="Images/Orbit.png" style="max-width: 70%; border-radius: 4px;">
</div>

# Part 2: NumPyro Crash Course

You have likely heard of other packages, such as `emcee` (an affine-invarient MCMC sampler, not gradient based), `Stan`/`PyStan` (good, but slower than NumPyro as it doesn't use Just-In-Time, JIT, complation) or `PyMC` (also lacks jit, doesn't have good native support for GPU stuff). We will use `NumPyro` because its fast, has automatic differentiation ("autodiff"), JIT by using `JAX`, and support for modern ML hardware.

In `NumPyro`, you specify your Bayesian model as a single function. This function encodes both the priors and likelihood within it. We will start simple with fitting a straight line:

## Generate Data

</div>

In [ ]:
# ==================
# Generate some data
# ==================
np.random.seed(42)

# True parameters
m_true = 2.0
b_true = 1.0

# Generate observations
n_obs = 30
x = np.linspace(0, 5, n_obs)
y_err = np.random.uniform(0.3, 1.5, n_obs)  # heteroscedastic uncertainties
y_obs = m_true * x + b_true + y_err * np.random.randn(n_obs)

# Convert to JAX arrays (NumPyro needs these)
x_jax     = jnp.array(x)
y_obs_jax = jnp.array(y_obs)
y_err_jax = jnp.array(y_err)

plt.figure(figsize=(8, 4))
plt.errorbar(x, y_obs, yerr=y_err, fmt='o', color='k', ecolor='grey', capsize=2, ms=4, label='Observations')
plt.plot(x, m_true * x + b_true, color='tab:blue', lw=1.5, alpha=0.7, label='True model')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.show()

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

## Defining the Model

</div>

In [ ]:
def model(x, y_obs=None, y_err=None):
    # Define Priors
    m = numpyro.sample('Slope', dist.Normal(0, 10))
    b = numpyro.sample('Intercept', dist.Normal(0, 10))

    # Model
    y_model = m*x + b

    # Track the model prediction at each data point
    numpyro.deterministic('y_model', y_model)

    # this is the log-likelihood
    numpyro.sample('obs', dist.Normal(y_model, y_err), obs = y_obs)

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

Let's go through this line by line

> `def model(x, y_obs=None, y_err=None):`

When defining our model function, we can do it basically however we want. We can choose how many arguments it should have. There is no minimum baseline, we don't have to pass `x`, `y_obs`, etc to it (although, you almost always will). The only thing that matters is being consistent. When we eventually want to run our mcmc, we will call `mcmc.run()`, and the arguments we pass to this must match the arguments we defined before. In this case, we would write `mcmc.run(jax.random.PRNGKey(0), x=x, y_obs=y_obs, y_err=y_err)`, which is the same order as above. We can include as many arguments as you want. For example, you could create a function like `def model(t, rv_obs=None, rv_err=None, n_planets=1, use_jitter=True):`

Fixed quantities that you don't want to infer (the time array, error bars, boolean flags) are just regular Python arguments. Only quantities declared with `numpyro.sample() inside the function body are treated as parameters to be inferred.

The convention of y_obs=None as a keyword argument with default None is deliberate, it lets you use the same model function for both inference (pass in data) and simulation (omit data). We'll see how this works when we get to the likelihood line below.

>  `m = numpyro.sample('Slope', dist.Normal(0, 10))`

This declares a model parameter and assigns it a prior distribution. The first argument tells `NumPyro` that `'Slope'` is a free parameter to be inferred, and the second argument specifies that the prior on this is a normal distribution with mean 0, and a standard deviation of 10. Each parameter must have a unique name. Give it something human-readable, as this wil be in outputs and whatnot. 

The distribution comes from `numpyro.distributions` (imported as `dist`). [You can read about the distributions here](https://num.pyro.ai/en/stable/distributions.html). Some commons ones take the following form:
- `dist.Normal(loc, scale)`
- `dist.Uniform(low, high)`
- `dist.HalfNormal(scale)`
- `dist.LogNormal(loc, scale)`
- `dist.TruncatedNormal(loc, scale, low, high)`
- `dist.Exponential(rate)`
- `dist.gamma(concentration, rate)`
- `dist.Poisson(rate)`

You can of course make your own custom distribution if you wish. [See this forum post for details (basically just copy the code of the regular distributions and rewrite it)](https://forum.pyro.ai/t/creating-a-custom-distribution-in-numpyro/3332).

During inference, `numpyro.sample()` returns the current value of the parameter as proposed by the sampler as a `JAX` scaler (or array), that you can use subsequent arithmetic as normal.

> `y_model = m * x + b`

This is standard arithmetic, and is the 'deterministic' part of the model (i.e. given parameter values $m$ and $b$, compute the predicted data). Because `m` and `b` are JAX-traced values, this line builds the 'computation graph' that JAX will later differentiate through (the autodiff stuff)

> `numpyro.deterministic('y_model', y_model)`

All this does is tell NumPyro to record `y_model` in the output samples, it is jut bookkeeping. This so when we eventually call `mcmc.get_samples()` in addition to telling us the values of `Slope` and `Intercept` it will tell us the value(s) of `y_model`. You can use this syntax for any derived quantity that you want (residuals, etc) rather than having to caluculate it after the fact.

> `numpyro.sample('obs', dist.Normal(y_model, y_err), obs=y_obs)`

Something that can be slightly confusing is the fact that the `.sample()` method is used for both the prior and the likelihood. The `obs` keyword argument changes its behaviour:

- Without `obs` (or `obs=None`), NumPyro draws a sample from the `dist.Normal(y_model, y_err)`, which isn't what we want. This would be a prior-predictive draw (i.e. given these parameters, generate a synthetic dataset).
- With `obs=y_obs`, NumPyro evalutates the log likelihood $\log \mathcal{L}(y_\text{obs})$ and adds it to the running log-probability. No sample is drawn.
- 
This is also why we set y_obs=None as the default in the function signature. When you call the model without data (for prior predictive checks), obs=None is passed through, and NumPyro draws synthetic observations instead of evaluating a likelihood. The same model function does double duty.

The `dist.Normal(y_model, y_err)` in this line is the noise model

Note that in this function, we don't `return` anything. Also note we do not calculaet the evidence.

## Running Inference

</div>

In [ ]:
# Choose a kernel (sampling algorithm, NUTS is a good choice)
kernel = NUTS(model)

# Set up the MCMC run
mcmc = MCMC(kernel, num_warmup=500, num_samples=1000, num_chains=4)

# Run it
mcmc.run(jax.random.PRNGKey(0), x, y_obs=y_obs, y_err=y_err, 
         extra_fields=('num_steps', 'energy', 'accept_prob'))

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

`num_warmup` is the adaption phase. NUTS uses this to tune the step size and the mass Matrix. These samples are discarded (they must be, so not to bias the results). `num_chains` runs independant walkers, more is better. You need several chains for many of the convergence diagnostics.

`jax.random.PRNGKey(0)` is JAX's random number generator. PRNG stands for pseudo-random number generator. The syntax for JAX's random stuff is somewhat different from numpys, so you might need to do some googling if you get stuck. 

The `extra_fields` tells MCMC to track some extra diagnostics - we will see these later. You don't include this at all.

## Extracting Results
</div>

In [ ]:
# Summary statistics (R-hat, n_eff, quantiles)
mcmc.print_summary()

# Raw samples as a dictionary: {'Intercept': 1D-array, 'Slope': 1D-array, 'y_model' : 2D-array}
samples = mcmc.get_samples()

extra_fields = mcmc.get_extra_fields()

# Divergences: must be zero
divergences = extra_fields['diverging'].sum()
print(f"Number of divergences: {divergences}")

# Acceptance probability: should be ~0.8 (NUTS default target)
print(f"Mean acceptance probability: {extra_fields['accept_prob'].mean():.3f}")

# Number of leapfrog steps per sample (NUTS adapts this automatically)
print(f"Leapfrog steps per sample: median={int(jnp.median(extra_fields['num_steps']))}, "
      f"max={int(extra_fields['num_steps'].max())}")


<div style="max-width: 800px; margin: 0 auto; text-align: justify;">
    
`print_summary()` gives you the essential diagnostics for each parameter:

- `mean`, `std`, and `median` are all straightforward
- `5.0%` and `95.0%` are the 5th and 95th quantiles. The range here is the 90% credible interval.
- `n_eff` the effective sample size (ESS). In total, we took 4000 steps (1000 steps times 4 chains), but because MCMC samples are correlated, the effective number of samples is lower (~800). This is a fine number.
- `r_hat` - This is the Gelman-Rubin convergence diagnostic. It compares the variance within each to to the variance between chains. If the chains have converged to the same distribution, $\hat{R} \approx 1$. The standard threshold is that $\hat{R} < 1.01$. If it is higher than 1.01, your chains haven't converged and you can't trust your results.

The extra fields that we printed off:

- `diverging` is the boolean array flagging samples where the leapfrog integrator diverged from the true Hamiltonian trajectory. This means the posterior geometry was too sharp for the step size. Must be zero. Non-zero divergences mean the posterior is likely being explored incorrectly, and your results may be biased.
- `accept_prob` is the acceptance probability at each step. NUTS targets ∼0.8\sim 0.8
∼0.8 by default (tuned during warm-up). If this is much lower, the sampler is struggling. The higher value here is fine because this straight line is very easy for HMC/NUTS.
- `num_steps`  is the number of leapfrog integration steps NUTS took per sample. NUTS adapts this automatically (the "No-U-Turn" criterion), so it varies from sample to sample. More steps means longer trajectories through parameter space. Very large values can indicate a difficult geometry. The leapfrog integration is similar to other physics numerical integrators that you probably learned, such as the Runge-Kutta-4 algorithm.
- `energy` is the Hamiltonian energy at each sample. Used for the energy diagnostic: if the distribution of energy transitions doesn't match the marginal energy distribution, the sampler isn't exploring efficiently. We'll use this later.

-----------

### Prior vs Posterior Predictive

A useful sanity check it to compare what the model predict before and after seeing the data. We can use NumPyro's `Predictive` to do this, we can just pass it the same model that we defined about for inference, to make predictions.

- Prior Predictive: This is basically pulling samples from the *prior*. I.e you are just seeing what models exist within your prior space. We can do this by calling: `Predictive(model, num_samples=200)`. What you are looking for is that these samples look something like what you think your data could look like. If it doesn't, your priors are likely misspecified.
- Posterior Predictive: Now that the model has been conditioned on the data, the samples we draw from the posterior are plausible models that fit our data. We can do this by calling: `Predictive(model, posterior_samples=samples)`. This is the important part. Your samples should reproduce the data fairly well.

The advantage of using Predictive rather than manually sampling and computing the forward model is that the model function is the single source of truth. If you change the priors, add a parameter, or modify the physics, the predictive checks update automatically.
Note: we pass a dummy `y_err` array here because the model signature requires it (it sets the width of the likelihood). Since we're only extracting `y_model` (the deterministic prediction, not noisy draws), the exact values don't affect the plotted curves.


</div>

In [ ]:
x_plot_jax = jnp.linspace(-0.5, 5.5, 200)

# Prior predictive: no posterior samples passed, draws from priors
prior_pred = Predictive(model, num_samples=200)(jax.random.PRNGKey(1), x_plot_jax, y_err=jnp.ones(200))

# Posterior predictive: pass posterior samples
post_pred = Predictive(model, posterior_samples=samples)(jax.random.PRNGKey(2), x_plot_jax, y_err=jnp.ones(200))

#############
# Plotting #
############
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for i in range(200):
    axes[0].plot(x_plot_jax, prior_pred['y_model'][i], color='tab:orange', lw=0.3, alpha=0.2)
    axes[1].plot(x_plot_jax, post_pred['y_model'][i], color='tab:blue', lw=0.3, alpha=0.2)

for ax in axes:
    ax.errorbar(x, y_obs, yerr=y_err, fmt='o', color='k', ecolor='grey', capsize=2, ms=4, zorder=5)
    ax.set_xlabel('x')

axes[0].set_ylim(-15, 25)
axes[0].set_ylabel('y')
axes[0].set_title('Prior Predictive')
axes[1].set_title('Posterior Predictive')
axes[0].plot([], [], color='tab:orange', lw=1, label='Prior draws')
axes[1].plot([], [], color='tab:blue', lw=1, label='Posterior draws')
axes[0].legend()
axes[1].legend()

plt.tight_layout()
plt.show()

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">
    
# Part 3: Two Planet RV Fit

We will now expand the problem from the pre-tutorial to include two planets (still on circular orbits). The radial velocity signal generated will be a superposition of two single planet signals:

$$v_r(t) = \gamma + K_1 \sin\left(\frac{2\pi t}{P_1} + \phi_1\right) +  K_2 \sin\left(\frac{2\pi t}{P_2} + \phi_2\right) $$

| Parameter | Symbol | True Value of Planet 1 | True Value of Planet 2|
|-----------|:------:|------------|----------------|
| System velocity | $\gamma$ | 3 m/s | - |
| Semi-amplitude of planet | $K_i$ | 4 m/s | 2 m/s |
| Period of planet| $P_i$ | 30 days | 40 days |
| Phase of planet| $\phi_i$ | 1.2 rad | 0.5 rad |

## Data Generation

</div>

In [ ]:
###################################
# Two-Planet RV — Data Generation #
###################################
np.random.seed(117)

# True parameters
gamma_true = 3.0      # m/s

K1_true    = 4.0      # m/s
P1_true    = 30.0     # days
phi1_true  = 1.2      # rad

K2_true    = 2.0      # m/s
P2_true    = 40.0     # days
phi2_true  = 0.5      # rad

# Two-planet circular RV model
def rv_two_planet(t, gamma, K1, P1, phi1, K2, P2, phi2):
    return (gamma
            + K1 * np.sin(2 * np.pi * t / P1 + phi1)
            + K2 * np.sin(2 * np.pi * t / P2 + phi2))

# Generate observations
n_obs = 100
t_obs = np.sort(np.random.uniform(0, 150, n_obs))
v_err = np.random.uniform(0.5, 1.5, n_obs)
v_true = rv_two_planet(t_obs, gamma_true, K1_true, P1_true, phi1_true,
                       K2_true, P2_true, phi2_true)
# Add noise
v_obs = v_true + v_err * np.random.randn(n_obs)

# Convert to JAX arrays
t_jax   = jnp.array(t_obs)
v_jax   = jnp.array(v_obs)
err_jax = jnp.array(v_err)

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">
    
## Plotting code

</div>

In [ ]:
t_fine = np.linspace(0, 150, 1000)
v_fine = rv_two_planet(t_fine, gamma_true, K1_true, P1_true, phi1_true,
                       K2_true, P2_true, phi2_true)

# Individual planet contributions
v1_fine = gamma_true + K1_true * np.sin(2 * np.pi * t_fine / P1_true + phi1_true)
v2_fine = gamma_true + K2_true * np.sin(2 * np.pi * t_fine / P2_true + phi2_true)

plt.figure(figsize=(10, 4))
plt.errorbar(t_obs, v_obs, yerr=v_err, fmt='o', color='k', ecolor='grey',
             capsize=2, ms=4, label='Observations')
plt.plot(t_fine, v_fine, color='tab:blue', lw=1.5, alpha=0.7, label='True model (combined)')
plt.plot(t_fine, v1_fine, color='tab:orange', lw=1, ls='--', alpha=0.5, label='Planet 1')
plt.plot(t_fine, v2_fine, color='tab:green', lw=1, ls='--', alpha=0.5, label='Planet 2')
plt.xlabel('Time [days]')
plt.ylabel('Radial velocity [m/s]')
plt.legend(loc = 'lower right')
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(10, 3))

# Panel 1: Lomb-Scargle periodogram
frequency, power = LombScargle(t_obs, v_obs, v_err).autopower(minimum_frequency=1/200, maximum_frequency=1/2)

periods = 1 / frequency

axes[0].plot(periods, power, color='k', lw=0.8)
axes[0].axvline(P1_true, color='tab:orange', ls='--', alpha=0.7, label=f'$P_1$ = {P1_true:.0f} d')
axes[0].axvline(P2_true, color='tab:green', ls='--', alpha=0.7, label=f'$P_2$ = {P2_true:.0f} d')
axes[0].set_xlabel('Period [days]')
axes[0].set_ylabel('Power')
axes[0].set_title('Lomb-Scargle')
axes[0].legend(fontsize=8)

# Panel 2: Phase-folded on P1
phase1 = (t_obs / P1_true) % 1
phase1_fine = np.linspace(0, 1, 500)
v1_phase = gamma_true + K1_true * np.sin(2 * np.pi * phase1_fine + phi1_true)

axes[1].errorbar(phase1, v_obs, yerr=v_err, fmt='o', color='k', ecolor='grey', capsize=2, ms=4)
axes[1].plot(phase1_fine, v1_phase, color='tab:orange', lw=1.5, alpha=0.7)
axes[1].set_xlabel('Phase')
axes[1].set_ylabel('Radial velocity [m/s]')
axes[1].set_title(f'Folded on $P_1$ = {P1_true:.0f} d')

# Panel 3: Phase-folded on P2
phase2 = (t_obs / P2_true) % 1
phase2_fine = np.linspace(0, 1, 500)
v2_phase = gamma_true + K2_true * np.sin(2 * np.pi * phase2_fine + phi2_true)

axes[2].errorbar(phase2, v_obs, yerr=v_err, fmt='o', color='k', ecolor='grey', capsize=2, ms=4)
axes[2].plot(phase2_fine, v2_phase, color='tab:green', lw=1.5, alpha=0.7)
axes[2].set_xlabel('Phase')
axes[2].set_ylabel('Radial velocity [m/s]')
axes[2].set_title(f'Folded on $P_2$ = {P2_true:.0f} d')

plt.tight_layout()
plt.show()


<div style="max-width: 800px; margin: 0 auto; text-align: justify;">
    
# The NumPyro Model  
**Exercise 1**: Complete the model below. You have 7 parameters to assign priors to. Think about appropriate bounds: semi-amplitudes must be positive, phases are angles on $[0, 2\pi)$, periods must be positive.


</div>

In [ ]:
def two_planet_model(t, v_obs=None, v_err=None):
    """
    NumPyro model for two circular-orbit planets.
    """

    # TODO: Define priors
    gamma = numpyro.sample('gamma', ...)
    
    K1    = numpyro.sample('K1',    ...)
    P1    = numpyro.sample('P1',    ...)
    phi1  = numpyro.sample('phi1',  ...)
    
    K2    = numpyro.sample('K2',    ...)
    P2    = numpyro.sample('P2',    ...)
    phi2  = numpyro.sample('phi2',  ...)

    # TODO: Model
    v_model =

    # TODO: Define the likelihood

<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">Hint 1: Priors</summary>

<div style="margin-top: 10px; padding: 15px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">

Think about the physical constraints on each parameter:

**$\gamma$ (system velocity):** Can be positive or negative as it's the bulk motion of the star along the line of sight. A broad `dist.Normal(0, 10)` is appropriate. This is weakly informative: it says we expect the system velocity to be within a few tens of m/s of zero, which is reasonable for nearby stars. It won't meaningfully constrain the fit unless the data are very poor.

**$K$ (semi-amplitudes):** Must be positive by definition (they are amplitudes). The simplest choice is `dist.Uniform(0, 20)`, but this creates hard walls at both boundaries that NUTS can bounce off. A better choice is `dist.HalfNormal(10)`, it's positive by construction, soft-edged, and gently regularises toward zero. This last property is physically motivated: if a planet doesn't exist, the posterior should naturally push $K \to 0$ rather than piling up against a hard lower bound. The scale of 10 m/s is generous enough to accommodate hot Jupiters while mildly disfavouring implausibly large values.

**$P$ (periods):** Must be positive, and you generally cannot reliably detect a period longer than your observing baseline. `dist.Uniform(1, 365)` is a reasonable default for a ~200-day dataset, it allows periods up to about a year. In a real analysis, you would often use a log-uniform (Jeffreys) prior, `dist.LogUniform(1, 365)`, which assigns equal prior weight per decade of period. This reflects genuine ignorance across orders of magnitude: before seeing the data, a 3-day period is no more or less plausible than a 300-day period. For this exercise, either works.

**$\phi$ (phases):** These are angles on $[0, 2\pi)$. `dist.Uniform(0, 2*jnp.pi)` is the correct uninformative prior — there is no physical reason to prefer any orbital phase over another. This is one of the rare cases where a uniform prior is genuinely the right choice, not just a lazy default.
</div>
</details>

<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">Hint 2: Solution</summary>

<div style="margin-top: 10px; padding: 15px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">
    
```python
def two_planet_model(t, v_obs=None, v_err=None):
    """
    NumPyro model for two circular-orbit planets.
    """

    gamma = numpyro.sample('gamma', dist.Normal(0, 10))
    
    K1   = numpyro.sample('K1',   dist.HalfNormal(10))       # positive, soft
    P1   = numpyro.sample('P1',   dist.Uniform(1, 150))      # up to ~1 year baseline
    phi1 = numpyro.sample('phi1', dist.Uniform(0, 2*jnp.pi))
    
    K2   = numpyro.sample('K2',   dist.HalfNormal(10))
    P2   = numpyro.sample('P2',   dist.Uniform(1, 150))
    phi2 = numpyro.sample('phi2', dist.Uniform(0, 2*jnp.pi))
    
    v_model = (gamma
               + K1 * jnp.sin(2 * jnp.pi * t / P1 + phi1)
               + K2 * jnp.sin(2 * jnp.pi * t / P2 + phi2))
    
    numpyro.deterministic('v_model', v_model)
    
    numpyro.sample('obs', dist.Normal(v_model, v_err), obs=v_obs)

```

</div>
</details>


<div style="max-width: 800px; margin: 0 auto; text-align: justify;">
    
Note that it is always a good idea to intialise near reasonable values, so that the MCMC needs less time to find the good modes. Remember that in high dimensions, random initialisation will almost never be in a "good" location. In our case, we could look at the Lomb-Scargle periodagram and get reasonable guesses for our periods, and with prior knowledge, we could make some reasonable guesses as to our parameter. To do so, when we choose our kernel, add in the following argument (I have chosen some reasonable numbers):

```python
        init_strategy = init_to_value(values={
            'gamma': 5.0,
            'K1':    5.0,
            'P1':    30.0,
            'phi1':  0.1,
            'K2':    5.0,
            'P2':    40.0,
            'phi2':  1.0,
        })
```

Note that for this problem, we need to be careful with choosing the periods, as there can be aliasing issues which give significant probability to periods that no planet is at. That's why we did a Lomb-Scargle periodogram first. Sometimes after the Lomb-Scargle you may even do an optimisation (say least squares) to get a better initial guess. You can be more flexible with the other parameters
</div>

In [ ]:
##################################################
# TODO: - Choose kernal (algorithm, go for NUTS) #
#       - initialise MCMC model                  #
#       - Run the model                          #
#       - Print summary                          #
##################################################

<div style="max-width: 800px; margin: 0 auto; text-align: justify; padding: 20px; background-color: rgba(0, 64, 133, 0.08); border: 2px solid rgba(0, 64, 133, 0.4); border-radius: 10px;">

<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">Solution</summary>

<div style="margin-top: 10px; padding: 15px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">

```python

# Choose a kernel (sampling algorithm, NUTS is a good choice)
kernel = NUTS(two_planet_model, init_strategy = init_to_value(values={
            'gamma': 5.0,
            'K1':    5.0,
            'P1':    30.0,
            'phi1':  0.1,
            'K2':    5.0,
            'P2':    40.0,
            'phi2':  1.0,
        }))

# Set up the MCMC run
mcmc = MCMC(kernel, num_warmup=500, num_samples=1000, num_chains=4)

# Run it
mcmc.run(jax.random.PRNGKey(0), t_jax, v_obs=v_jax, v_err=err_jax,
         extra_fields=('num_steps', 'energy', 'accept_prob'))
         
```

</div>
</details>
</div

In [ ]:
# Corner plot
samples = mcmc.get_samples()
chain_arr = np.column_stack([
    np.array(samples['gamma']),
    np.array(samples['K1']),
    np.array(samples['P1']),
    #np.array(samples['phi1']), # <--------- we don't care about phase
    np.array(samples['K2']),
    np.array(samples['P2']),
   # np.array(samples['phi2']), # <--------- we don't care about phase
])

labels = [r'$\gamma$', r'$K_1$', r'$P_1$',
          r'$K_2$', r'$P_2$']
truths = [gamma_true, K1_true, P1_true, 
          K2_true, P2_true]

fig = corner.corner(chain_arr, labels=labels, truths=truths,
                    truth_color='black', show_titles=True)

<div style="max-width: 800px; margin: 0 auto; text-align: justify; padding: 20px; background-color: rgba(0, 64, 133, 0.08); border: 2px solid rgba(0, 64, 133, 0.4); border-radius: 10px;">

**Exercise 2: Posterior predictive check.**

Draw 100 random samples from the posterior and overplot the corresponding model curves on the data. Does the model reproduce the data?


For each step of the Metropolis algorithm, identify which line(s) of the implementation above correspond to it. Try to answer before revealing.

<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">Solution</summary>

<div style="margin-top: 10px; padding: 15px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">

```python

t_plot = np.linspace(0, 150, 1000)
plt.figure(figsize=(10, 4))
idx = np.random.choice(len(samples['gamma']), 100, replace=False)

for i in idx:
    g  = float(samples['gamma'][i])
    k1 = float(samples['K1'][i])
    p1 = float(samples['P1'][i])
    f1 = float(samples['phi1'][i])
    k2 = float(samples['K2'][i])
    p2 = float(samples['P2'][i])
    f2 = float(samples['phi2'][i])
    v = (g + k1np.sin(2np.pit_plot/p1 + f1)
    + k2np.sin(2np.pit_plot/p2 + f2))
plt.plot(t_plot, v, color='tab:blue', lw=0.3, alpha=0.1)
plt.plot([], [], color='tab:blue', lw=1, alpha=0.5, label='Posterior samples')
plt.errorbar(t_obs, v_obs, yerr=v_err, fmt='o', color='k', ecolor='grey',
capsize=2, ms=4, zorder=5, label='Observations')
plt.xlabel('Time [days]')
plt.ylabel('Radial velocity [m/s]')
plt.legend()
plt.show()
```

</div>
</details>

**Exercise 3: The effect of initialisation.**

- a) Rerun the sampler changing the initial guess for all the parameters except the periods. Does anything change?
- b) Rerun the sampler changing the initial guess only for the periods. What happens? Why?

<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">Elaborate</summary>

<div style="margin-top: 10px; padding: 15px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">

For a), the posterior should look pretty similar, as these parameters are essentially unimodal (unless you initialised far enough away for the labels to switch.

For b), you should see multi-modality. Some chains may have converged to different modes, producing the multimodal structure in the combined samples. This is evident in summary output as the $\hat{R}$ values are greater than 1.01. The primary peak should be close to the true values, but the secondary modes are period aliases (integer or rational-rato multiples of the true period). These are genuine features of the posterior, and aree not sampling artifacts. The elevated $\hat{R}$ values tell you that the chains have not agreed on a single mode. This is HMC/NUTS telling you that the posterior is multimodal.

**An important limitation of MCMC:** HMC/NUTS is designed to efficiently explore a single mode. It follows the gradient along the typical set, which is a local operation. It cannot reliably jump between well-separated modes because the posterior density in between is essentially zero, and no gradient-based trajectory will cross that gap. If your posterior is genuinely multimodal (as periodic problems almost always are), you need different tools:

- **Parallel tempering** heats up the posterior (flattening the modes) so chains can jump between them, then cools back down.
- **Nested sampling** (e.g., `dynesty`, which you used in the previous tutorial) explores the full prior volume and naturally handles multimodality.
  
</div>
</details>

**Exercise 4: Redce the data.**

Set `n_obs = 12` and regenerate the data. Re-run NUTS with good initialisation. How does the posterior change? Can NUTS still recover both planets?

</div>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">
    
# Part 4: Diagnostics

For the following section, we will be comparing the results of a "good" initialisation and a "bad initialisation". It is a good idea to actually look at the diagnostics between them. The good initialisation is:
``` python
    init_to_value(values={'gamma': 5.0, 'K1': 5.0, 'P1': 30.0, 'phi1':  0.1,
                                        'K2': 5.0, 'P2': 40.0, 'phi2':  1.0})))
```

The bad initialisation is:
``` python
    init_to_value(values={'gamma': 5.0, 'K1': 5.0, 'P1': 35.0, 'phi1':  0.1,
                                        'K2': 5.0, 'P2': 45.0, 'phi2':  1.0})))
```


So far we have seen some diagnostics by calling `print_summary()`, which showed things like $\hat{R}$, ESS, and number of divergences. Here we will look at some visual diagnostics that can be useful. We will be using the `ArviZ` library which interacts with `NumPyro` to easily create some nice plots.

First, we need to covert from the `NumPyro` format to the `ArviZ` format. Fortunately, there is a one-liner that we can use:

</div>

In [ ]:
# (this cell assumes `mcmc` from Part 3 is still in memory)
inference_data = az.from_numpyro(mcmc)
print(inference_data)

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

`inference_data` is an `ArviZ` object that just contains, unsurprisinly, the pamrater samples, the diagnostics, the log-likelihood, and the observed data. This is just a nice way to store the data that makes the following plotting code very easily. You can make all of the following plots manually if you want to.

### Trace plots

The trace plot is the first thing (and probably most important thing) you should look at. Each row is a parameter. The left panel shows the marginal posterior (i.e. the histogram for that parameter), and the right panel shows the *trace*, the value of that parameter at each step, with one linestyle per chain (if `compact = True`) or one colour per chain (if `compact = False`).

For the "good" initilisation, you should see:
- Left panel: all chains more or less have the same distribution.
- Right panel: it should look like a "hairy caterpillar". This means that the chain bounces rapidly around a stationary mean, that should overlap.

For the "bad" intialisation, you might see:
- Left panel: histograms that have different shapes, or the same shape at different locations
- Right panel: "hairy caterpillars" that don't overlap


</div>

In [ ]:
az.plot_trace(inference_data, var_names=['gamma', 'K1', 'P1', 'K2', 'P2'], compact = False)
plt.tight_layout()
plt.show()

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

### Rank plots

I personally have never seen this used, but it is another validation metric. It is apparently a more robust alternative to trace plots. The idea is to:
- 1) Pool all samples from all chains for a given parameter
- 2) Replace each with value with its rank in that combined pool (i.e. sort position 1 to $N_\text{total}$)
- 3) Split back by chain and histogram the ranks.
 
If all chains are sampling the same distribution (which they should be, ideally) each chain contributes roughly equally to every part of the rank ordering, so its rank histogram is flat (uniform). If a chain is stuck at systematically higher or lower values (i.e. at a different mode), its ranks will pile up near the top or bottom. 

These are meant to be better than trace plots because trace plots require you to visually judge whether the overlapping wiggly lines are "close enough", which can be subjective. Ranking reduces the uncertainty by just making you check if the histograms are flat or not. 

As before, look at how the histograms look for both the "good" and "bad" initialisations.
</div>

In [ ]:
az.plot_rank(inference_data, var_names=['gamma', 'K1', 'P1', 'K2', 'P2'])
plt.tight_layout()
plt.show()

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

### Autocorrelation

By construction, MCMC samples are correlated (less so for HMC/NUTS). The autocorrelation function tells you how many steps it takes for the correlation to drop off. HMC should decorrelate fast. Other algorithms are slower. While it is not inherently bad to have a long autocorrelation fuction, it means that it takes more steps to get independant samples. 

The below plot shows the autocorrelation of each parameter (row) for each chain (column). Likewise, check how this differs for "good" vs "bad" initialisations.
</div>

In [ ]:
az.plot_autocorr(inference_data, var_names=['gamma', 'K1', 'P1', 'K2', 'P2'], max_lag=50)
plt.tight_layout()
plt.show()

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

### Corner-plot

`ArviZ` has a built in corner-plot. We have seen these before. Personally, I prefer the `corner` corner plots, but it is good to know that this exists.
</div>

In [ ]:
az.plot_pair(inference_data, var_names=['gamma', 'K1', 'P1', 'K2', 'P2'],
            divergences=True, kind='scatter', marginals=True)
plt.show()

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">
    
# Part 5: Cephied Variables
*Note: I didn't have time to come up with this problem entirely myself. The extra scatter stuff is my work, but applying it specifically to Cephied Variables was helped with Claude AI. As such, there is a decent chance that the actual equations could be wrong. Even if that is the case, the point of this section still stands.*

Classic Cephied Variables obey a tight period-luminosity relation, that has been useful as a cosmic ladder rung. The relationship is approximately linear in log-period:
$$M = a \log_{10}(P/\text{days}) + b $$
where $M$ is the absolute magnitude and $P$ is the pulsation period. 

We will fit this relationship with a slight twist. We suspect that there is an intrinsic scatter, $\sigma_{\text{int}}$, on top of the measurement uncertainties. This could be something like stellar metallicity or other things that affect this relationship that we are not explicitly modelling. The total variance at each point is then:
$$ \sigma^2_{\text{tot},i} = \sigma^2_{\text{obs},i} + \sigma^2_{\text{int}}$$

We will fit for $\sigma_{\text{int}}$ as a free parameter (which is a standard practice, if you think your errors are unestimated). If there is no intrinsic scatter, the posterior for $\sigma_\text{int}$ will pile up near zero and you have lost nothing (except computer cycles). If it is nonzero, ignoring it would give you overconfident uncertainties on $a$ and $b$.

### Generate Data
</div>

In [ ]:
#######################################
# Cepheid P-L relation - Simulated Data
#######################################
np.random.seed(161)

# True parameters
a_true         = -2.76   # slope (mag per dex)
b_true         = -1.40   # zero-point (mag at P = 1 day, i.e. log P = 0)
sigma_int_true = 0.12  # intrinsic scatter (mag)

# Generate log-periods roughly spanning the Cepheid instability strip
n_ceph = 50
logP = np.random.uniform(0.4, 1.8, n_ceph)  # log10(P/days), ~2.5 to ~63 day Cepheids

# True absolute magnitudes on the PL relation + intrinsic scatter
M_true = a_true * logP + b_true + sigma_int_true * np.random.randn(n_ceph)

# Observational uncertainties (heteroscedastic)
M_err = np.random.uniform(0.05, 0.25, n_ceph)
M_obs = M_true + M_err * np.random.randn(n_ceph)

# JAX arrays
logP_jax  = jnp.array(logP)
M_obs_jax = jnp.array(M_obs)
M_err_jax = jnp.array(M_err)

# Plot
plt.figure(figsize=(8, 5))
plt.errorbar(logP, M_obs, yerr=M_err, fmt='o', color='k', ecolor='grey',
             capsize=2, ms=4, label='Observations')
logP_fine = np.linspace(0.3, 1.9, 200)
plt.plot(logP_fine, a_true * logP_fine + b_true, 'tab:blue', lw=1.5, label='True PL relation')
plt.gca().invert_yaxis()
plt.xlabel(r'$\log_{10}(P/\mathrm{days})$')
plt.ylabel(r'Absolute magnitude $M$')
plt.legend()
plt.title('Simulated Cepheid Period-Luminosity Relation')
plt.show()

<div style="max-width: 800px; margin: 0 auto; text-align: justify; padding: 20px; background-color: rgba(0, 64, 133, 0.08); border: 2px solid rgba(0, 64, 133, 0.4); border-radius: 10px;">

**Exercise 5:** Complete the `NumPyro` model for the Cepheid relation. You have three parameters: slope $a$, intercept $b$, and intrinsic scatter $\sigma_\text{int}$

<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">Hint 1</summary>

<div style="margin-top: 10px; padding: 15px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">
    
The intrinsic scatter must be positive. `dist.HalfNormal` is a good choice.

</div>
</details>

<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">Hint 2</summary>

<div style="margin-top: 10px; padding: 15px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">
    
The total uncertainty at each point is:
    $$ \sigma_{\text{tot},i} = \sqrt{\sigma^2_{\text{obs},i} + \sigma^2_{\text{int}}} $$


</div>
</details>

<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">Hint 3</summary>

<div style="margin-top: 10px; padding: 15px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">
    
The likelihood uses the total uncertainty, not just the observationalm uncertainty.

</div>
</details>



</div>

In [ ]:
def cepheid_pl_model(logP, M_obs=None, M_err=None):
    # TODO: Priors

    # TODO: Model prediction

    # TODO: Total uncertainty (obs + intrinsic in quadrature)

    # TODO: Likelihood


<div style="max-width: 800px; margin: 0 auto; text-align: justify; padding: 20px; background-color: rgba(0, 64, 133, 0.08); border: 2px solid rgba(0, 64, 133, 0.4); border-radius: 10px;">

<details>
<summary>Solution</summary>

```python
def cepheid_pl_model(logP, M_obs=None, M_err=None):
    a         = numpyro.sample('a', dist.Normal(-3, 2))
    b         = numpyro.sample('b', dist.Normal(0, 5))
    sigma_int = numpyro.sample('sigma_int', dist.HalfNormal(1.0))

    M_model   = a * logP + b
    sigma_tot = jnp.sqrt(M_err**2 + sigma_int**2)
    
    numpyro.deterministic('M_model', M_model)
    numpyro.sample('obs', dist.Normal(M_model, sigma_tot), obs=M_obs)
```

</details>


</div>

In [ ]:
#########
# TODO: #
#########

# Run MCMC

# initialise MCMC

# run MCMC

# print summary

<div style="max-width: 800px; margin: 0 auto; text-align: justify; padding: 20px; background-color: rgba(0, 64, 133, 0.08); border: 2px solid rgba(0, 64, 133, 0.4); border-radius: 10px;">

<details>
<summary>Solution</summary>

Something like this:
 
```python
    kernel = NUTS(cepheid_pl_model)
    mcmc_ceph = MCMC(kernel, num_warmup=500, num_samples=2000, num_chains=4)
    mcmc_ceph.run(jax.random.PRNGKey(1), logP_jax, M_obs=M_obs_jax, M_err=M_err_jax)
    mcmc_ceph.print_summary()
```

</details>


</div>

In [ ]:
#########
# TODO: #
#########

# Make a corner plot

<div style="max-width: 800px; margin: 0 auto; text-align: justify; padding: 20px; background-color: rgba(0, 64, 133, 0.08); border: 2px solid rgba(0, 64, 133, 0.4); border-radius: 10px;">

<details>
<summary>Solution</summary>

Something like this:
 
```python
samp_ceph = mcmc_ceph.get_samples()
chain_ceph = np.column_stack([
    np.array(samp_ceph['a']),
    np.array(samp_ceph['b']),
    np.array(samp_ceph['sigma_int']),
])

fig = corner.corner(chain_ceph,
                    labels=[r'$a$ (slope)', r'$b$ (intercept)', r'$\sigma_\mathrm{int}$'],
                    truths=[a_true, b_true, sigma_int_true],
                    truth_color='black', show_titles=True)
plt.show()
```

</details>


</div>

In [ ]:
#########
# TODO: #
#########

# plot posterior samples with a 1 sigma shaded region

<div style="max-width: 800px; margin: 0 auto; text-align: justify; padding: 20px; background-color: rgba(0, 64, 133, 0.08); border: 2px solid rgba(0, 64, 133, 0.4); border-radius: 10px;">

<details>
<summary>Solution</summary>

Something like this:
 
```python
logP_plot = np.linspace(0.3, 1.9, 200)
plt.figure(figsize=(8, 5))
idx = np.random.choice(len(samp_ceph['a']), 200, replace=False)
for i in idx:
    ai = float(samp_ceph['a'][i])
    bi = float(samp_ceph['b'][i])
    si = float(samp_ceph['sigma_int'][i])
    M_line = ai * logP_plot + bi
    plt.plot(logP_plot, M_line, color='tab:blue', lw=0.3, alpha=0.05)
    
plt.fill_between(logP_plot, M_line - si, M_line + si,
color='tab:red', alpha=0.1)
plt.errorbar(logP, M_obs, yerr=M_err, fmt='o', color='k', ecolor='grey',
             capsize=2, ms=4, zorder=5, label='Observations')
plt.gca().invert_yaxis()
plt.xlabel(r'$\log_{10}(P/\mathrm{days}$')
plt.ylabel(r'Absolute magnitude $M$')
plt.title('Posterior Predictive: Cepheid PL Relation')
plt.legend()
plt.show()
```

</details>


</div>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">
    
# Part 6: Galaxy Surface Brightness

We will simulate a 32x32 pixel image of an elliptical galaxy. We will do so assuming a 2D elliptical Gaussian (yes yes, I know that this is very simplistic - I am not a galaxy person). We will add noise, and fit it using NUTS. The model is:

$$I(x,y) = A \exp \left[-\frac{1}{2}\left(\frac{x'^2}{\sigma_x^2} + \frac{y'^2}{\sigma_y^2}\right)\right] + B $$

where the rotated coordinates are:
$$\begin{align}
x' &= (x-x_0)\cos\theta + (y-y_0)\sin\theta \\
y' &= -(x-x_0)\sin\theta + (y-y_0)\cos\theta
\end{align}$$

The parameters are then:
- centroid $(x_0, y_0)$,
- amplitude $A$,
- widths $(\sigma_x, \sigma_y)$
- position angle $\theta$
- Background $B$.

That's 7 in total.

### Generate Data
First, let's generate a synthetic observation. The code below creates the pixel grid, defines the true parameters, generates the noiseless model, and adds Gaussian noise.

</div>

In [ ]:
np.random.seed(0)

# Pixel grid
img_size = 32
y_grid, x_grid = np.mgrid[0:img_size, 0:img_size]

# True parameters
x0_true    = 16.3
y0_true    = 15.7
A_true     = 120.0
sx_true    = 3.0
sy_true    = 5.0
theta_true = 30.0   # degrees
B_true     = 10.0

noise_sigma = 8.0


def gaussian_2d(x, y, x0, y0, A, sx, sy, theta_deg):
    """Elliptical Gaussian in pixel coordinates."""
    theta = np.radians(theta_deg)
    dx = x - x0
    dy = y - y0

    x_rot =  dx * np.cos(theta) + dy * np.sin(theta)
    y_rot = -dx * np.sin(theta) + dy * np.cos(theta)

    r2 = (x_rot / sx)**2 + (y_rot / sy)**2
    return A * np.exp(-0.5 * r2)


true_image = gaussian_2d(
    x_grid, y_grid,
    x0_true, y0_true,
    A_true, sx_true, sy_true,
    theta_true
) + B_true

obs_image = true_image + noise_sigma * np.random.randn(img_size, img_size)

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">
    
### Plot
</div>

In [ ]:
# Display
vmin, vmax = np.percentile(obs_image, [1, 99])
plt.imshow(obs_image, origin='lower', cmap='magma', vmin=vmin, vmax=vmax)
plt.title("Observed Image")
plt.colorbar()
plt.show()

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">
    
### Convert to JAX Arrays
NumPyro runs on JAX, so we need JAX versions of our grids and data. We also need a JAX-compatible version of the 2D Gaussian model (replace `np` with `jnp`).


</div>

In [ ]:
x_jax = jnp.array(x_grid.astype(float))
y_jax = jnp.array(y_grid.astype(float))
obs_flat = jnp.array(obs_image.ravel())


def gaussian_2d_jax(x, y, x0, y0, A, sx, sy, theta_deg):
    """JAX version of the elliptical Gaussian."""
    theta = jnp.radians(theta_deg)
    dx = x - x0
    dy = y - y0

    x_rot =  dx * jnp.cos(theta) + dy * jnp.sin(theta)
    y_rot = -dx * jnp.sin(theta) + dy * jnp.cos(theta)

    r2 = (x_rot / sx)**2 + (y_rot / sy)**2
    return A * jnp.exp(-0.5 * r2)

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

## Write the NumPyro Model

Write a NumPyro probabilistic model that:

1. Samples the 7 parameters from appropriate priors
2. Constructs the model image using `gaussian_2d_jax`
3. Defines the likelihood as a Gaussian with known noise $\sigma = 8.0$

Some guidance on priors:

- For the centroid $(x_0, y_0)$: a Normal prior centred near the image centre is reasonable. We roughly know where the source is.
- For positive-definite parameters like $A$, $\sigma_x$, $\sigma_y$: sample in log-space (i.e. sample `log_A` from a Normal, then compute $A = e^{\text{log\_A}}$). This keeps them positive and gives the sampler an unbounded parameter to work with.
- For $\theta$: try a simple approach for now. Sample an unconstrained parameter `theta_uncon` from a Normal(0, 1), then transform: $\theta = (\texttt{theta\_uncon} \times 90) \mod 180$.
- For $B$: a Normal prior centred on your rough estimate of the background.

</div>

In [ ]:
def psf_model(x, y, obs_flat=None, noise=8.0):
    """
    NumPyro model for a 2D elliptical Gaussian PSF.

    Parameters to sample:
        x0, y0     : centroid
        log_A      : log amplitude
        log_sx     : log sigma_x
        log_sy     : log sigma_y
        theta_uncon: unconstrained angle parameter
        B          : background level
    """

    # ^---- Use the above label names 
    
    # --- Centroid ---
    # TODO: Sample x0 and y0 from Normal priors

    # --- Amplitude (log-space) ---
    # TODO: Sample log_A, then compute A = exp(log_A)

    # --- Widths (log-space) ---
    # TODO: Sample log_sx and log_sy, then exponentiate

    # --- Angle ---
    # TODO: Sample theta_uncon, then compute theta = (theta_uncon * 90) % 180

    # --- Background ---
    # TODO: Sample B

    # --- Forward model ---
    # TODO: Compute model image using gaussian_2d_jax, add B

    # --- Likelihood ---
    # TODO: numpyro.sample("obs", dist.Normal(model.ravel(), noise), obs=obs_flat)
    pass

<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: #cce5ff; border: 1px solid #004085; border-radius: 5px; color: #004085;">Hint 1: Priors</summary>

<div style="margin-top: 10px; padding: 15px; background-color: #cce5ff; border: 1px solid #004085; border-radius: 5px; color: #004085;">

For log-space parameters, you want something like:

```python
log_A = numpyro.sample("log_A", dist.Normal(np.log(100.0), 1.0))
A = jnp.exp(log_A)
```

The prior on `log_A` is centred at $\ln(100) \approx 4.6$, with a standard deviation of 1.0 in log-space. This corresponds to a lognormal prior on $A$ that covers roughly one order of magnitude either side of 100.

</div>
</details>

<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: #cce5ff; border: 1px solid #004085; border-radius: 5px; color: #004085;">Hint 2: Full Solution</summary>

<div style="margin-top: 10px; padding: 15px; background-color: #cce5ff; border: 1px solid #004085; border-radius: 5px; color: #004085;">

```python
def psf_model(x, y, obs_flat=None, noise=8.0):

    x0 = numpyro.sample("x0", dist.Normal(16.0, 3.0))
    y0 = numpyro.sample("y0", dist.Normal(16.0, 3.0))

    log_A = numpyro.sample("log_A", dist.Normal(np.log(100.0), 1.0))
    A = jnp.exp(log_A)

    log_sx = numpyro.sample("log_sx", dist.Normal(np.log(4.0), 0.5))
    log_sy = numpyro.sample("log_sy", dist.Normal(np.log(4.0), 0.5))
    sx = jnp.exp(log_sx)
    sy = jnp.exp(log_sy)

    theta_uncon = numpyro.sample("theta_uncon", dist.Normal(0.0, 1.0))
    theta = (theta_uncon * 90.0) % 180.0

    B = numpyro.sample("B", dist.Normal(10.0, 5.0))

    model_img = gaussian_2d_jax(x, y, x0, y0, A, sx, sy, theta) + B

    numpyro.sample("obs",
                   dist.Normal(model_img.ravel(), noise),
                   obs=obs_flat)
```

</div>
</details>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

## Run NUTS

Run the sampler with 4 chains, 1000 warmup steps, and 2000 samples per chain. Use `target_accept_prob=0.9`.

</div>

In [ ]:
#################
# TODO RUN NUTS #
#################

<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: #cce5ff; border: 1px solid #004085; border-radius: 5px; color: #004085;">Solution</summary>

<div style="margin-top: 10px; padding: 15px; background-color: #cce5ff; border: 1px solid #004085; border-radius: 5px; color: #004085;">

```python
kernel = NUTS(psf_model, target_accept_prob=0.9)

mcmc = MCMC(kernel,
            num_warmup=1000,
            num_samples=2000,
            num_chains=4)

mcmc.run(jax.random.PRNGKey(1),
         x_jax, y_jax,
         obs_flat=obs_flat,
         noise=noise_sigma)

mcmc.print_summary()
```

</div>
</details>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

## Make a Corner Plot

Extract the samples, transform back to physical parameters (exponentiate log-space parameters, convert `theta_uncon` to degrees), and make a corner plot with the true values overlaid.

</div>

In [ ]:
############
# TODO:   #
###########

# Extract samples with mcmc.get_samples()

# Transform: A = exp(log_A), sx = exp(log_sx), etc.

# theta = (theta_uncon * 90) % 180

# Stack into array, make corner plot with truths overlaid
#
# samples = mcmc.get_samples()
# ...
# corner.corner(chain, labels=[...], truths=[...], truth_color="tab:red", show_titles=True)

<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: #cce5ff; border: 1px solid #004085; border-radius: 5px; color: #004085;">Solution</summary>

<div style="margin-top: 10px; padding: 15px; background-color: #cce5ff; border: 1px solid #004085; border-radius: 5px; color: #004085;">

```python
samples = mcmc.get_samples()

A_samp     = np.exp(samples["log_A"])
sx_samp    = np.exp(samples["log_sx"])
sy_samp    = np.exp(samples["log_sy"])
theta_samp = (np.array(samples["theta_uncon"]) * 90.0) % 180.0

chain = np.column_stack([
    samples["x0"],
    samples["y0"],
    A_samp,
    sx_samp,
    sy_samp,
    theta_samp,
    samples["B"]
])

labels = [r"$x_0$", r"$y_0$", r"$A$",
          r"$\sigma_x$", r"$\sigma_y$",
          r"$\theta$", r"$B$"]

truths = [x0_true, y0_true, A_true,
          sx_true, sy_true,
          theta_true, B_true]

corner.corner(chain,
              labels=labels,
              truths=truths,
              truth_color="tab:red",
              show_titles=True)
plt.show()
```

</div>
</details>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

## Residual Check

Compute the posterior MAP model and inspect the residuals. A good fit should leave residuals consistent with noise.

</div>

In [ ]:
A_samp     = np.exp(np.array(samples["log_A"]))
sx_samp    = np.exp(np.array(samples["log_sx"]))
sy_samp    = np.exp(np.array(samples["log_sy"]))
theta_samp = (np.array(samples["theta_uncon"]) * 90.0) % 180.0
x0_samp    = np.array(samples["x0"])
y0_samp    = np.array(samples["y0"])
B_samp     = np.array(samples["B"])

# Find the sample with the lowest chi-squared
chi2 = np.zeros(len(x0_samp))
for i in range(len(x0_samp)):
    m = gaussian_2d(x_grid, y_grid,
                    x0_samp[i], y0_samp[i],
                    A_samp[i], sx_samp[i], sy_samp[i],
                    theta_samp[i]) + B_samp[i]
    chi2[i] = np.sum(((obs_image - m) / noise_sigma)**2)

best_idx = np.argmin(chi2)

best_model = gaussian_2d(
    x_grid, y_grid,
    x0_samp[best_idx], y0_samp[best_idx],
    A_samp[best_idx], sx_samp[best_idx], sy_samp[best_idx],
    theta_samp[best_idx]
) + B_samp[best_idx]

residual = obs_image - best_model

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(obs_image, origin='lower', cmap='magma', vmin=vmin, vmax=vmax)
axes[0].set_title("Observed")

axes[1].imshow(best_model, origin='lower', cmap='magma', vmin=vmin, vmax=vmax)
axes[1].set_title("Best-Fit Model")

im = axes[2].imshow(residual, origin='lower', cmap='RdBu_r', vmin=-20, vmax=20)
axes[2].set_title("Residual")
plt.colorbar(im, ax=axes[2])

plt.tight_layout()
plt.show()

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

## Discussion: Why are the $\hat{R}$ Values Bad?

You should see output like this:

```
              mean       std    median      5.0%     95.0%     n_eff     r_hat
        B      9.96      0.32      9.96      9.46     10.49   7232.84      1.00
    log_A      4.78      0.01      4.78      4.76      4.80   5233.17      1.00
   log_sx      1.47      0.23      1.60      1.07      1.63      2.01     16.29
   log_sy      1.21      0.23      1.09      1.05      1.62      2.01     16.67
theta_uncon    0.58      2.17      0.83     -2.68      3.34      2.00    201.07
       x0     16.28      0.05     16.28     16.20     16.36   6313.93      1.00
       y0     15.72      0.06     15.72     15.61     15.82   7584.91      1.00
```

Most parameters look great: $\hat{R} \approx 1.0$, thousands of effective samples. But `log_sx`, `log_sy`, and `theta_uncon` have catastrophic $\hat{R}$ values (16, 17, 201!) and $n_{\text{eff}} \approx 2$. Yet the residual plot looks fine. What is going on?

**Question 1:** Look at the corner plot carefully. What do you notice about $\sigma_x$ and $\sigma_y$? Why might $\hat{R}$ be terrible even though the model fits well?

**Question 2:** Why did we plot MAP instead of the mean model? What would happen if we plotted the mean model?


</div>

<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: #cce5ff; border: 1px solid #004085; border-radius: 5px; color: #004085;">Answer</summary>

<div style="margin-top: 10px; padding: 15px; background-color: #cce5ff; border: 1px solid #004085; border-radius: 5px; color: #004085;">

There are two distinct problems here, both rooted in **parameter degeneracies** and **symmetry**.

**1. Label Swapping ($\sigma_x \leftrightarrow \sigma_y$):**

The model is invariant under the simultaneous exchange $\sigma_x \leftrightarrow \sigma_y$ and $\theta \to \theta + 90°$. An ellipse with $\sigma_x = 3$, $\sigma_y = 5$, $\theta = 30°$ is identical to one with $\sigma_x = 5$, $\sigma_y = 3$, $\theta = 120°$. The posterior therefore has (at least) two symmetric modes.

Different chains can settle into different modes. Chain 1 might find ($\sigma_x \approx 3$, $\sigma_y \approx 5$, $\theta \approx 30°$) while Chain 2 finds ($\sigma_x \approx 5$, $\sigma_y \approx 3$, $\theta \approx 120°$). Each chain is internally consistent, but when $\hat{R}$ compares the between-chain variance to the within-chain variance, it sees massive disagreement. Hence $\hat{R} \gg 1$.

This is a **multimodal posterior**. NUTS is a gradient-based sampler, so it excels at characterising a single mode but cannot jump between well-separated modes. This isn't a failure of the sampler; the posterior genuinely has this symmetry.

**2. The Angle Parameterisation:**

The transformation $\theta = (\texttt{theta\_uncon} \times 90) \bmod 180$ maps a continuous, unbounded parameter to a periodic one via a modular operation. The problem is that HMC/NUTS uses gradients, and the modulus operation has a discontinuous derivative (it's piecewise linear with jumps). The sampler sees a flat landscape in `theta_uncon` with occasional cliffs at the wraparound points. This makes it very hard to explore efficiently.

**The fix for both problems** is reparameterisation, which we will address in Part 2.

**2:**

Due to the multimodal posterior, the mean is in a region of low probability (i.e. a bad fit). 

</div>
</details>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

---

# Part 2: Contaminated Images — Galaxy + Star

## Motivation

In real imaging data, the source you care about is rarely alone. Foreground stars, neighbouring galaxies, cosmic rays, and detector artefacts all contaminate your images. A common strategy is to mask the contamination and fit only the unmasked pixels.

In this exercise, we will simulate a galaxy (elliptical Gaussian, as before) contaminated by a bright foreground star with diffraction spikes. We will then fit the galaxy model with and without a mask, and see how the contamination biases the inferred parameters.

We also fix the parameterisation problems from Part 1.

## What Changes in the Model?

Before writing code, let's discuss three changes to the galaxy model compared to Part 1.

### Change 1: Breaking the Label-Swap Degeneracy

In Part 1, we sampled $\sigma_x$ and $\sigma_y$ independently. This created a symmetric bimodal posterior because swapping ($\sigma_x \leftrightarrow \sigma_y$, $\theta \to \theta + 90°$) gives an identical model.

The fix is to enforce an ordering by parameterising in terms of the semi-major axis $\sigma_{\text{major}}$ and the axis ratio $q = \sigma_{\text{minor}} / \sigma_{\text{major}} \leq 1$:

$$\sigma_x = \sigma_{\text{major}} \cdot q, \quad \sigma_y = \sigma_{\text{major}}$$

Since $q \leq 1$ by construction, $\sigma_x \leq \sigma_y$ always. There is no second mode to confuse the chains.

In practice, we sample `log_s_major` and `log_ratio` (both unbounded), then enforce $q \leq 1$ with `jnp.minimum(jnp.exp(log_ratio), 1.0)`.

### Change 2: Fixing the Angle

Instead of the `(theta_uncon * 90) % 180` hack, we now sample $\theta$ directly from a Uniform(0, $\pi/2$) prior. The range $[0, \pi/2]$ is sufficient because the label-swap fix removes the $\theta \leftrightarrow \theta + 90°$ degeneracy, and an ellipse is symmetric under $\theta \to \theta + \pi$.

Note: NumPyro handles the bounded-to-unbounded transformation for Uniform distributions automatically under the hood, so the sampler still works in unconstrained space.

### Change 3: Masking via `numpyro.factor`

Normally, a NumPyro model defines the likelihood through a `numpyro.sample("obs", ..., obs=data)` call. This adds $\sum_i \ln p(d_i \mid \theta)$ to the log-posterior.

To mask pixels, we need finer control. Instead of the `obs` pattern, we compute the per-pixel log-probabilities ourselves, zero out the masked pixels, and inject the total into the model using `numpyro.factor`:

```python
logp = dist.Normal(model, noise).log_prob(obs)
numpyro.factor("masked_loglike", jnp.sum((1 - mask) * logp))
```

Where `mask` is 1 for contaminated pixels and 0 otherwise. This way, masked pixels contribute nothing to the likelihood.

### Generate Data

</div>

In [ ]:
np.random.seed(4)

img_size = 32
y_grid, x_grid = np.mgrid[0:img_size, 0:img_size]

# ----- True Galaxy Parameters -----
x0_true, y0_true = 16.0, 15.0
A_true     = 120.0
sx_true    = 3.0    # minor axis
sy_true    = 5.0    # major axis
theta_true = np.deg2rad(25.0)  # radians
B_true     = 10.0
noise_sigma = 10.0

# ----- Star Parameters -----
star_x, star_y = 22.0, 18.0
star_amp   = 150.0
star_sigma = 0.8
spike_amp  = 100.0
spike_decay = 6.0


def gaussian_2d(x, y, x0, y0, A, sx, sy, theta):
    """Elliptical Gaussian. theta in radians."""
    dx = x - x0
    dy = y - y0
    x_rot =  dx * np.cos(theta) + dy * np.sin(theta)
    y_rot = -dx * np.sin(theta) + dy * np.cos(theta)
    r2 = (x_rot / sx)**2 + (y_rot / sy)**2
    return A * np.exp(-0.5 * r2)


# Galaxy
gal = gaussian_2d(x_grid, y_grid,
                  x0_true, y0_true,
                  A_true, sx_true, sy_true,
                  theta_true)

# Star: core + diffraction spikes
dxs = x_grid - star_x
dys = y_grid - star_y
r_star = np.sqrt(dxs**2 + dys**2)

star_core = star_amp * np.exp(-0.5 * (r_star / star_sigma)**2)

d1 = np.abs(dxs - dys) / np.sqrt(2)
d2 = np.abs(dxs + dys) / np.sqrt(2)
spikes = spike_amp * (
    np.exp(-d1**2 / (2 * 0.7**2)) +
    np.exp(-d2**2 / (2 * 0.7**2))
) * np.exp(-r_star / spike_decay)

star_total = star_core + spikes

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

### Plot Data

</div>

In [ ]:
# Observed image
true_image = gal + star_total + B_true
obs_image  = true_image + noise_sigma * np.random.randn(img_size, img_size)

plt.imshow(obs_image, origin='lower', cmap='magma')
plt.title("Observed (Galaxy + Star)")
plt.colorbar()
plt.show()

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

### Mask out the star and diffraction spikes

Construct a mask that covers the star core and its diffraction spikes. The mask is `True` (1) for contaminated pixels and `False` (0) for clean pixels.

1) Mask out all pixels within 3 pixels of the star itslef
2) Mask out the spikes within 1 pixel of the diagonals, and within 14 pixels of the star core
</div>

In [ ]:
mask_core   = # TODO
mask_spikes = # TODO
mask = mask_core | mask_spikes

plt.imshow(mask, origin='lower')
plt.title("Mask")
plt.colorbar()
plt.show()

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: #cce5ff; border: 1px solid #004085; border-radius: 5px; color: #004085;">Answer</summary>

<div style="margin-top: 10px; padding: 15px; background-color: #cce5ff; border: 1px solid #004085; border-radius: 5px; color: #004085;">
    
```python
mask_core   = r_star < 3.0
mask_spikes = ((d1 < 1.0) | (d2 < 1.0)) & (r_star < 14)
mask = mask_core | mask_spikes
```
</div>
</details>

### Convert to JAX Arrays

</div> 

In [ ]:
x_jax    = jnp.array(x_grid.astype(float))
y_jax    = jnp.array(y_grid.astype(float))
obs_jax  = jnp.array(obs_image)
mask_jax = jnp.array(mask.astype(float))


def gaussian_2d_jax(x, y, x0, y0, A, sx, sy, theta):
    """JAX elliptical Gaussian. theta in radians."""
    dx = x - x0
    dy = y - y0
    x_rot =  dx * jnp.cos(theta) + dy * jnp.sin(theta)
    y_rot = -dx * jnp.sin(theta) + dy * jnp.cos(theta)
    r2 = (x_rot / sx)**2 + (y_rot / sy)**2
    return A * jnp.exp(-0.5 * r2)

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

## Step 4: Write the Galaxy Model

Write a NumPyro model that:

1. Samples the galaxy parameters with the improved parameterisation ($\sigma_{\text{major}}$, axis ratio $q$, $\theta \sim \text{Uniform}(0, \pi/2)$)
2. Uses `numpyro.factor` with the mask to zero out contaminated pixels

The model should accept `x`, `y`, `obs` (2D image), `mask` (2D, float), and `noise` as arguments.

</div>

In [ ]:
def galaxy_model(x, y, obs, mask, noise=10.0):
    """
    NumPyro model for an elliptical Gaussian galaxy.

    Improved parameterisation:
        x0, y0        : centroid
        log_A         : log amplitude
        log_s_major   : log semi-major axis width
        log_ratio     : log axis ratio (q = minor/major, capped at 1)
        theta         : position angle, Uniform(0, pi/2)
        B             : background

    Uses numpyro.factor for masked likelihood.
    """

    # --- Centroid ---
    # TODO

    # --- Amplitude ---
    # TODO: Sample log_A, exponentiate

    # --- Size: major axis + axis ratio ---
    # TODO: Sample log_s_major and log_ratio
    # s_major = exp(log_s_major)
    # ratio   = min(exp(log_ratio), 1.0)
    # sx = s_major * ratio    (minor axis)
    # sy = s_major            (major axis)

    # --- Angle ---
    # TODO: Sample theta from Uniform(0, pi/2)

    # --- Background ---
    # TODO

    # --- Forward model ---
    # TODO: model = gaussian_2d_jax(...) + B

    # --- Masked likelihood ---
    # TODO: Compute per-pixel log-probs, multiply by (1-mask), sum, add via numpyro.factor
    pass

<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: #cce5ff; border: 1px solid #004085; border-radius: 5px; color: #004085;">Hint 1: The Axis Ratio</summary>

<div style="margin-top: 10px; padding: 15px; background-color: #cce5ff; border: 1px solid #004085; border-radius: 5px; color: #004085;">

```python
log_s_major = numpyro.sample("log_s_major", dist.Normal(np.log(4.5), 0.5))
log_ratio   = numpyro.sample("log_ratio",   dist.Normal(np.log(0.6), 0.5))

s_major = jnp.exp(log_s_major)
ratio   = jnp.minimum(jnp.exp(log_ratio), 1.0)  # enforce q <= 1

sx = s_major * ratio   # minor axis
sy = s_major           # major axis
```

The prior on `log_ratio` is centred at $\ln(0.6) \approx -0.5$, which says "we expect the galaxy to be somewhat elongated." The `jnp.minimum` clips the ratio to be at most 1, which enforces $\sigma_x \leq \sigma_y$.

</div>
</details>

<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: #cce5ff; border: 1px solid #004085; border-radius: 5px; color: #004085;">Hint 2: The Masked Likelihood</summary>

<div style="margin-top: 10px; padding: 15px; background-color: #cce5ff; border: 1px solid #004085; border-radius: 5px; color: #004085;">

```python
logp = dist.Normal(model, noise).log_prob(obs)
numpyro.factor("masked_loglike", jnp.sum((1 - mask) * logp))
```

`dist.Normal(model, noise).log_prob(obs)` computes the per-pixel Gaussian log-probability. Multiplying by `(1 - mask)` zeroes out contaminated pixels. `numpyro.factor` adds the scalar total to the log-posterior.

</div>
</details>

<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: #cce5ff; border: 1px solid #004085; border-radius: 5px; color: #004085;">Hint 3: Full Solution</summary>

<div style="margin-top: 10px; padding: 15px; background-color: #cce5ff; border: 1px solid #004085; border-radius: 5px; color: #004085;">

```python
def galaxy_model(x, y, obs, mask, noise=10.0):

    x0 = numpyro.sample("x0", dist.Normal(16.0, 3.0))
    y0 = numpyro.sample("y0", dist.Normal(16.0, 3.0))

    log_A = numpyro.sample("log_A", dist.Normal(np.log(100), 1.0))
    A = jnp.exp(log_A)

    log_s_major = numpyro.sample("log_s_major",
                                 dist.Normal(np.log(4.5), 0.5))
    log_ratio = numpyro.sample("log_ratio",
                               dist.Normal(np.log(0.6), 0.5))

    s_major = jnp.exp(log_s_major)
    ratio = jnp.minimum(jnp.exp(log_ratio), 1.0)

    sx = s_major * ratio
    sy = s_major

    theta = numpyro.sample("theta",
                           dist.Uniform(0.0, jnp.pi / 2.0))

    B = numpyro.sample("B", dist.Normal(10, 5))

    model = gaussian_2d_jax(x, y, x0, y0, A, sx, sy, theta) + B
    logp = dist.Normal(model, noise).log_prob(obs)

    numpyro.factor("masked_loglike",
                   jnp.sum((1 - mask) * logp))
```

</div>
</details>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

## Fit WITHOUT the Mask

First, let's see what happens when we naively fit the galaxy model to the contaminated image *without* masking the star. Pass `jnp.zeros_like(mask_jax)` as the mask argument (i.e. no pixels are masked).

</div>

In [ ]:
# TODO: Run NUTS with galaxy_model, passing mask=jnp.zeros_like(mask_jax)

# kernel_nomask = 
# mcmc_nomask   = 

# print("\n=== WITHOUT MASK ===")
# mcmc_nomask.print_summary()

<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: #cce5ff; border: 1px solid #004085; border-radius: 5px; color: #004085;">Solution</summary>

<div style="margin-top: 10px; padding: 15px; background-color: #cce5ff; border: 1px solid #004085; border-radius: 5px; color: #004085;">

```python
kernel_nomask = NUTS(galaxy_model, target_accept_prob=0.9)
mcmc_nomask = MCMC(kernel_nomask,
                   num_warmup=1500,
                   num_samples=2000,
                   num_chains=4)

mcmc_nomask.run(jax.random.PRNGKey(0),
                x_jax, y_jax,
                obs_jax,
                jnp.zeros_like(mask_jax),
                noise=noise_sigma)

print("\n=== WITHOUT MASK ===")
mcmc_nomask.print_summary()
```

</div>
</details>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

## Fit WITH the Mask

Now fit using the actual mask.

</div>

In [ ]:
# TODO: Run NUTS with galaxy_model, now passing mask=mask_jax

# kernel_mask =
# mcmc_mask   =

# print("\n=== WITH MASK ===")
# mcmc_mask.print_summary()

<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: #cce5ff; border: 1px solid #004085; border-radius: 5px; color: #004085;">Solution</summary>

<div style="margin-top: 10px; padding: 15px; background-color: #cce5ff; border: 1px solid #004085; border-radius: 5px; color: #004085;">

```python
kernel_mask = NUTS(galaxy_model, target_accept_prob=0.9)
mcmc_mask = MCMC(kernel_mask,
                 num_warmup=1500,
                 num_samples=2000,
                 num_chains=4)

mcmc_mask.run(jax.random.PRNGKey(1),
              x_jax, y_jax,
              obs_jax,
              mask_jax,
              noise=noise_sigma)

print("\n=== WITH MASK ===")
mcmc_mask.print_summary()
```

</div>
</details>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

## Diagnostics and Comparison

Now let's compare the two fits. The code below transforms the samples, computes posterior mean models, makes residual plots, corner plots, and overlays the 1D marginal posteriors.

Run this cell and inspect the outputs.

</div>

In [ ]:
samples_nomask = mcmc_nomask.get_samples(group_by_chain=True)
samples_mask   = mcmc_mask.get_samples(group_by_chain=True)

# True values in model parameterisation
true_vals = {
    "x0": x0_true,
    "y0": y0_true,
    "A":  A_true,
    "sx": min(sx_true, sy_true),
    "sy": max(sx_true, sy_true),
    "theta": theta_true,
    "B":  B_true
}


def transform(samples):
    """Transform from sampled to physical parameters."""
    s = {k: np.array(v) for k, v in samples.items()}

    A = np.exp(s["log_A"])
    s_major = np.exp(s["log_s_major"])
    ratio = np.minimum(np.exp(s["log_ratio"]), 1.0)

    sx = s_major * ratio
    sy = s_major

    return {
        "x0": s["x0"],
        "y0": s["y0"],
        "A":  A,
        "sx": sx,
        "sy": sy,
        "theta": s["theta"],
        "B": s["B"]
    }


def flatten(d):
    return {k: v.reshape(-1) for k, v in d.items()}


flat_nomask = flatten(transform(samples_nomask))
flat_mask   = flatten(transform(samples_mask))

In [ ]:
# --- Posterior Mean Models and Residuals ---

def posterior_mean_model(flat):
    m = {k: np.mean(v) for k, v in flat.items()}
    return gaussian_2d(
        x_grid, y_grid,
        m["x0"], m["y0"],
        m["A"], m["sx"], m["sy"],
        m["theta"]
    ) + m["B"]


model_nomask = posterior_mean_model(flat_nomask)
model_mask   = posterior_mean_model(flat_mask)

resid_nomask = obs_image - model_nomask
resid_mask   = obs_image - model_mask

vmin, vmax = np.percentile(obs_image, [1, 99])

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

axes[0, 0].imshow(obs_image, origin='lower', cmap='magma', vmin=vmin, vmax=vmax)
axes[0, 0].set_title("Observed")

axes[0, 1].imshow(model_nomask, origin='lower', cmap='magma', vmin=vmin, vmax=vmax)
axes[0, 1].set_title("Model (No Mask)")

axes[0, 2].imshow(resid_nomask, origin='lower', cmap='RdBu_r', vmin=-20, vmax=20)
axes[0, 2].set_title("Residual (No Mask)")

axes[1, 0].imshow(obs_image, origin='lower', cmap='magma', vmin=vmin, vmax=vmax)
axes[1, 0].contour(mask, colors='cyan', linewidths=0.5)
axes[1, 0].set_title("Observed + Mask")

axes[1, 1].imshow(model_mask, origin='lower', cmap='magma', vmin=vmin, vmax=vmax)
axes[1, 1].set_title("Model (Masked)")

axes[1, 2].imshow(resid_mask, origin='lower', cmap='RdBu_r', vmin=-20, vmax=20)
axes[1, 2].set_title("Residual (Masked)")

plt.tight_layout()
plt.show()

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

### Mean Model vs MAP

In the above cell, we plotted the *mean model*. In the previous part, we instead plotted the MAP. What is the difference? Go back and also plot the MAP in this case

</div>

In [ ]:
# --- Trace Plots ---

az.plot_trace(az.from_numpyro(mcmc_nomask))
plt.suptitle("Trace \u2013 No Mask")
plt.show()

az.plot_trace(az.from_numpyro(mcmc_mask))
plt.suptitle("Trace \u2013 Masked")
plt.show()

In [ ]:
# --- Corner Plots ---

labels = ["x0", "y0", "A", "sx", "sy", "theta", "B"]
truth_vector = [true_vals[k] for k in labels]

corner.corner(
    np.column_stack([flat_nomask[k] for k in labels]),
    labels=labels,
    truths=truth_vector,
    truth_color="red",
    show_titles=True
)
plt.suptitle("Posterior \u2013 No Mask")
plt.show()

corner.corner(
    np.column_stack([flat_mask[k] for k in labels]),
    labels=labels,
    truths=truth_vector,
    truth_color="red",
    show_titles=True
)
plt.suptitle("Posterior \u2013 Masked")
plt.show()

In [ ]:
# --- 1D Posterior Comparison ---

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
axes = axes.flatten()

for i, param in enumerate(labels):
    axes[i].hist(flat_nomask[param], bins=50,
                 density=True, alpha=0.5, label="No Mask")
    axes[i].hist(flat_mask[param], bins=50,
                 density=True, alpha=0.5, label="Masked")
    axes[i].axvline(true_vals[param],
                    color='red', linestyle='--', linewidth=2)
    axes[i].set_title(param)

axes[0].legend()
axes[-1].axis('off')
plt.tight_layout()
plt.show()

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

## Discussion: How Does the Star Affect the Fit?

You should see results like:

**Without mask:**
```
           mean       std    median      5.0%     95.0%     n_eff     r_hat
    B     10.40      0.43     10.39      9.65     11.07   4507.27      1.00
log_A      4.80      0.01      4.80      4.78      4.83   4007.72      1.00
log_ratio  0.25      0.23      0.19     -0.02      0.57   2043.69      1.00
log_s_major 1.51     0.01      1.51      1.49      1.53   3481.78      1.00
theta      0.78      0.46      0.78      0.00      1.41   6248.94      1.00
   x0     17.43      0.07     17.43     17.32     17.54   5156.05      1.00
   y0     15.51      0.06     15.51     15.40     15.62   4923.65      1.00
```

**With mask:**
```
           mean       std    median      5.0%     95.0%     n_eff     r_hat
    B     10.78      0.41     10.78     10.11     11.44   8545.18      1.00
log_A      4.81      0.02      4.81      4.78      4.84   7125.63      1.00
log_ratio -0.48      0.03     -0.48     -0.53     -0.44   6902.04      1.00
log_s_major 1.59     0.02      1.59      1.56      1.62   5848.56      1.00
theta      0.44      0.03      0.44      0.39      0.48   9313.47      1.00
   x0     16.07      0.06     16.07     15.97     16.18   9453.01      1.00
   y0     14.79      0.08     14.79     14.65     14.93   8569.36      1.00
```

**Question:** Compare these two fits. What has the star done to the unmasked fit, and why?

**Question:** Despite the model without the mask performing poorly, it has an `r_hat` of 1 across the board, and fairly high `n_eff`. Why? What are these diagnostics actually telling us?
</div>

<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: #cce5ff; border: 1px solid #004085; border-radius: 5px; color: #004085;">Answer</summary>

<div style="margin-top: 10px; padding: 15px; background-color: #cce5ff; border: 1px solid #004085; border-radius: 5px; color: #004085;">

Notice that both fits converge ($\hat{R} \approx 1$), both have good $n_{\text{eff}}$. The sampler hasn't failed. It has converged  to the wrong answer in the unmasked case. This is arguably worse than a failed fit, because it gives you no warning.

Look at what changed:

- **Centroid ($x_0$):** Shifted from 16.0 (true) to 17.4. The star at $x = 22$ is pulling the centroid toward it. The model is trying to capture both the galaxy and the star with a single Gaussian, so the centroid moves to a compromise position between them.

- **Axis ratio (`log_ratio`):** Went from $-0.48$ (masked, corresponding to $q \approx 0.62$, a genuine elongation) to $+0.25$ (unmasked, corresponding to $q \approx 1.3$, clipped to 1.0). The unmasked fit thinks the galaxy is nearly circular. Why? The star adds extra flux on one side of the galaxy. The model compensates by making the Gaussian rounder and shifting the centroid toward the star.

- **Position angle ($\theta$):** 0.78 rad (unmasked) vs 0.44 rad (masked). The true value is $0.436$ rad ($25°$). The unmasked fit's angle is biased because the star distorts the apparent orientation.

- **Semi-major axis (`log_s_major`):** 1.51 (unmasked) vs 1.59 (masked). The unmasked fit finds a slightly smaller major axis, consistent with being rounder (since the model can't account for the elongated flux from the star+galaxy combination).

The residual plot makes this clear: in the unmasked case, you'll see strong residual structure where the star is, and the model itself is visibly displaced from the galaxy centre. The masked fit, by ignoring the contaminated pixels, recovers parameters close to the truth.

This is a general lesson: contamination doesn't just add noise, it introduces systematic bias. The sampler will happily converge to a precise but wrong answer. Masking (or modelling the contaminant) is essential.

</div>
</details>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

## When High SNR Breaks Things

Go back to the simulation cell and **decrease the noise**. Try `noise_sigma = 3.0` instead of 10.0 (i.e. increase the SNR significantly). Re-run both fits.

**Question:** What happens to the unmasked fit? Why does higher SNR make things *worse*?

</div>

In [ ]:
# TODO: Re-run the simulation with noise_sigma = 3.0
# Then re-run both fits (with and without mask)
# Inspect the print_summary() output

<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: #cce5ff; border: 1px solid #004085; border-radius: 5px; color: #004085;">Answer</summary>

<div style="margin-top: 10px; padding: 15px; background-color: #cce5ff; border: 1px solid #004085; border-radius: 5px; color: #004085;">

At high noise ($\sigma = 10$), the unmasked fit converges cleanly ($\hat{R} \approx 1$) but to biased values. At low noise ($\sigma = 3$), you will likely see $\hat{R} \gg 1$ and low $n_{\text{eff}}$ for the unmasked fit.

Why? It comes down to model misspecification and the likelihood.

When the noise is high, the star is partially hidden in the noise. The likelihood surface is broad, and the single-Gaussian model can find a "compromise" fit that doesn't perfectly match either the galaxy or the star, but isn't catastrophically wrong anywhere. The $\chi^2$ penalty for the misfit at the star's location is modest because $\sigma$ is large. The sampler converges to this compromise smoothly.

When the noise is low, the star stands out sharply. The residuals at the star's location become enormous relative to $\sigma$. Each pixel of star contamination contributes a huge $\chi^2$ penalty ($\propto (\text{residual}/\sigma)^2$). The likelihood surface becomes extremely sensitive to the star: small changes in the galaxy parameters produce large changes in the log-likelihood at the star pixels. This creates a likelihood landscape with sharp features that NUTS struggles to navigate. Different chains may find different local compromises, leading to poor $\hat{R}$.

The masked fit should remain well-behaved at both noise levels, because it never sees the star pixels.

The lesson: **higher SNR amplifies model misspecification**. If your model is wrong (here, a single Gaussian for a galaxy + star), more data doesn't help — it makes the problem more apparent. At low SNR, you can get away with an imperfect model. At high SNR, you can't.

</div>
</details>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

---------------

# Further Reading
I have read the first three, and they are particularly good. The others are apparently good too.


- **Betancourt (2018)**, "A Conceptual Introduction to HMC"
- **Hogg & Foreman-Mackey (2018)**, "Data Analysis Recipes: Using MCMC"
- **Sharma (2017)**, "MCMC Methods for Bayesian Data Analysis in Astronomy"
- Gelman et al. (2014), *Bayesian Data Analysis* (3rd ed.)
- MacKay (2003), *Information Theory, Inference, and Learning Algorithms*
- Sivia & Skilling (2006), *Data Analysis: A Bayesian Tutorial*
- Hoffman & Gelman (2014), "The No-U-Turn Sampler"
- Foreman-Mackey et al. (2013), "emcee: The MCMC Hammer"

</div>